In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm import tqdm

In [2]:
#模型v1

# ========== 配置 ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", DEVICE)

NUM_CLASSES = 5
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 30
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
N_FOLDS = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ========== 路径 ==========
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
WORK_DIR = "/kaggle/working"

# ========== 加载训练数据 ==========
def load_train_data(data_dir):
    filenames, labels = [], []
    classes = sorted(os.listdir(data_dir))
    for label, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for fname in os.listdir(cls_dir):
            filenames.append(os.path.join(cls_dir, fname))
            labels.append(label)
    return filenames, labels, classes

train_files, train_labels, class_names = load_train_data(TRAIN_DIR)
print(f"总样本数: {len(train_files)}")
print(f"类别: {class_names}")
print(f"分布: {pd.Series(train_labels).value_counts().to_dict()}")

# ========== 数据增强 ==========
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ========== Dataset ==========
class PlantDataset(Dataset):
    def __init__(self, filenames, labels, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        img = Image.open(self.filenames[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# ========== 模型 ==========
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

# ========== 单折训练 ==========
def train_one_fold(fold, train_f, train_l, val_f, val_l):
    print(f"\n===== Fold {fold} =====")
    train_ds = PlantDataset(train_f, train_l, train_transform)
    val_ds = PlantDataset(val_f, val_l, val_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # 阶段1：冻结backbone
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=1e-4)
    for epoch in range(EPOCHS_HEAD):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Head] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    # 阶段2：解冻微调
    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE)

    best_f1, best_state, wait = 0, None, 0
    for epoch in range(EPOCHS_FINETUNE):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Fine] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE)
                p = model(imgs).argmax(dim=1).cpu().numpy()
                preds.extend(p); gts.extend(labels.numpy())
        macro_f1 = f1_score(gts, preds, average='macro')
        micro_f1 = f1_score(gts, preds, average='micro')
        print(f"Epoch {epoch+1}: Macro={macro_f1:.4f}, Micro={micro_f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= 5:
                print("早停")
                break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"{WORK_DIR}/model_fold{fold}.pth")
    print(f"Fold {fold} 最佳 Macro-F1: {best_f1:.4f}")
    return best_f1

# ========== 5折训练 ==========
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(train_files, train_labels)):
    tr_f = [train_files[i] for i in tr_idx]
    tr_l = [train_labels[i] for i in tr_idx]
    va_f = [train_files[i] for i in val_idx]
    va_l = [train_labels[i] for i in val_idx]
    fold_scores.append(train_one_fold(fold, tr_f, tr_l, va_f, va_l))

print(f"\n========== 5折平均 Macro-F1: {np.mean(fold_scores):.4f} ==========")

使用设备: cuda
总样本数: 500
类别: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']
分布: {0: 100, 1: 100, 2: 100, 3: 100, 4: 100}

===== Fold 0 =====
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]
[Fine] E1: 100%|██████████| 25/25 [00:04<00:00,  5.84it/s]


Epoch 1: Macro=0.7067, Micro=0.7200


[Fine] E2: 100%|██████████| 25/25 [00:04<00:00,  5.51it/s]


Epoch 2: Macro=0.8232, Micro=0.8300


[Fine] E3: 100%|██████████| 25/25 [00:04<00:00,  5.85it/s]


... (105 outputs left out)

[Fine] E11: 100%|██████████| 25/25 [00:04<00:00,  5.41it/s]


Epoch 11: Macro=0.9000, Micro=0.9000


[Fine] E12: 100%|██████████| 25/25 [00:04<00:00,  5.32it/s]


Epoch 12: Macro=0.8995, Micro=0.9000
早停
Fold 4 最佳 Macro-F1: 0.9198

========== 5折平均 Macro-F1: 0.8900 ==========


In [3]:
# ========== 预测 + TTA + 生成 submission（自包含版本）==========
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

# ========== 配置 ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 5
IMG_SIZE = 224
N_FOLDS = 5
WORK_DIR = "/kaggle/working"

# ========== 路径 ==========
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")

# ========== 类别名（必须和训练时顺序一致）==========
class_names = sorted(os.listdir(TRAIN_DIR))
print("类别顺序:", class_names)

# ========== 验证集 transform ==========
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ========== 模型构建（预测时不需要下载预训练权重）==========
def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

# ========== TTA 预测 ==========
def predict_with_tta(model, files):
    model.eval()
    probs = []
    with torch.no_grad():
        for f in tqdm(files, desc="Predict"):
            img = Image.open(f).convert("RGB")
            x = val_transform(img).unsqueeze(0).to(DEVICE)
            # 原图 + 水平翻转
            p1 = F.softmax(model(x), dim=1)
            p2 = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            probs.append(((p1 + p2) / 2).cpu().numpy())
    return np.concatenate(probs, axis=0)

# ========== 加载测试集文件名 ==========
test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)])
print(f"测试集数量: {len(test_files)}")

# ========== 5 折模型集成 + TTA ==========
ensemble_probs = np.zeros((len(test_files), NUM_CLASSES))
for fold in range(N_FOLDS):
    model_path = f"{WORK_DIR}/model_fold{fold}.pth"
    model = build_model()
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    probs = predict_with_tta(model, test_files)
    ensemble_probs += probs
    print(f"Fold {fold} 预测完成")
ensemble_probs /= N_FOLDS
preds = ensemble_probs.argmax(axis=1)

# ========== 生成 submission（严格匹配示例格式）==========
submission = pd.DataFrame({
    "ID": [os.path.basename(f) for f in test_files],
    "Category": [class_names[i] for i in preds]
})
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)

# ========== 检查输出 ==========
print("\n========== 提交文件检查 ==========")
print(submission.head(10))
print(f"\n总行数: {len(submission)}")
print(f"列名: {list(submission.columns)}")
print(f"\n类别分布:\n{submission['Category'].value_counts()}")
print(f"\n文件已保存到: {WORK_DIR}/submission.csv")

类别顺序: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']
测试集数量: 378


Predict: 100%|██████████| 378/378 [00:11<00:00, 33.39it/s]


Fold 0 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 52.86it/s]


Fold 1 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.55it/s]


Fold 2 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.86it/s]


Fold 3 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 52.76it/s]

Fold 4 预测完成

========== 提交文件检查 ==========
               ID           Category
0  abkieurfrr.png         Sugar beet
1  ablzkpojla.png         Sugar beet
2  acmbgjflct.png   Loose Silky-bent
3  afhcnytjtd.png   Loose Silky-bent
4  ahoyeyvosg.png   Loose Silky-bent
5  aijmdaocym.png        Black-grass
6  ajjwrhipgh.png   Loose Silky-bent
7  apjrnavrnr.png         Sugar beet
8  arkzcnxddq.png  Scentless Mayweed
9  armpwctjip.png         Sugar beet

总行数: 378
列名: ['ID', 'Category']

类别分布:
Category
Loose Silky-bent     107
Scentless Mayweed     92
Sugar beet            78
Black-grass           59
Common wheat          42
Name: count, dtype: int64

文件已保存到: /kaggle/working/submission.csv


In [4]:
# ========== 完整训练脚本 v2：延长微调 + Mixup ==========
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm import tqdm

# ========== 配置 ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", DEVICE)

NUM_CLASSES = 5
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 50      # ← 从 30 延长到 50
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 10             # ← 从 5 放宽到 10
MIXUP_PROB = 0.5          # ← 每个 batch 有 50% 概率用 mixup
MIXUP_ALPHA = 0.2
N_FOLDS = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ========== 路径 ==========
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
WORK_DIR = "/kaggle/working"

# ========== 加载训练数据 ==========
def load_train_data(data_dir):
    filenames, labels = [], []
    classes = sorted(os.listdir(data_dir))
    for label, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for fname in os.listdir(cls_dir):
            filenames.append(os.path.join(cls_dir, fname))
            labels.append(label)
    return filenames, labels, classes

train_files, train_labels, class_names = load_train_data(TRAIN_DIR)
print(f"总样本数: {len(train_files)}")
print(f"类别: {class_names}")
print(f"分布: {pd.Series(train_labels).value_counts().to_dict()}")

# ========== 数据增强 ==========
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ========== Dataset ==========
class PlantDataset(Dataset):
    def __init__(self, filenames, labels, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        img = Image.open(self.filenames[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# ========== 模型 ==========
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

# ========== Mixup ==========
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# ========== 单折训练 ==========
def train_one_fold(fold, train_f, train_l, val_f, val_l):
    print(f"\n===== Fold {fold} =====")
    train_ds = PlantDataset(train_f, train_l, train_transform)
    val_ds = PlantDataset(val_f, val_l, val_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ---------- 阶段1：冻结 backbone ----------
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=1e-4)

    for epoch in range(EPOCHS_HEAD):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Head] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    # ---------- 阶段2：解冻微调 + Mixup ----------
    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE)

    best_f1, best_state, wait = 0, None, 0
    for epoch in range(EPOCHS_FINETUNE):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Fine] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            # 随机决定是否使用 Mixup
            if np.random.rand() < MIXUP_PROB:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                outputs = model(imgs)
                loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
            else:
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
        scheduler.step()

        # ---------- 验证 ----------
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE)
                p = model(imgs).argmax(dim=1).cpu().numpy()
                preds.extend(p); gts.extend(labels.numpy())
        macro_f1 = f1_score(gts, preds, average='macro')
        micro_f1 = f1_score(gts, preds, average='micro')
        print(f"Epoch {epoch+1}: Macro={macro_f1:.4f}, Micro={micro_f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print("早停")
                break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"{WORK_DIR}/model_v2_fold{fold}.pth")
    print(f"Fold {fold} 最佳 Macro-F1: {best_f1:.4f}")
    return best_f1

# ========== 5 折训练 ==========
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(train_files, train_labels)):
    tr_f = [train_files[i] for i in tr_idx]
    tr_l = [train_labels[i] for i in tr_idx]
    va_f = [train_files[i] for i in val_idx]
    va_l = [train_labels[i] for i in val_idx]
    fold_scores.append(train_one_fold(fold, tr_f, tr_l, va_f, va_l))

print(f"\n========== v2 五折平均 Macro-F1: {np.mean(fold_scores):.4f} ==========")

使用设备: cuda
总样本数: 500
类别: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']
分布: {0: 100, 1: 100, 2: 100, 3: 100, 4: 100}

===== Fold 0 =====


[Fine] E1: 100%|██████████| 25/25 [00:04<00:00,  5.70it/s]


Epoch 1: Macro=0.7158, Micro=0.7300


[Fine] E2: 100%|██████████| 25/25 [00:04<00:00,  5.52it/s]


Epoch 2: Macro=0.8265, Micro=0.8300


[Fine] E3: 100%|██████████| 25/25 [00:04<00:00,  5.74it/s]


... (287 outputs left out)

[Fine] E34: 100%|██████████| 25/25 [00:04<00:00,  5.45it/s]


Epoch 34: Macro=0.8996, Micro=0.9000


[Fine] E35: 100%|██████████| 25/25 [00:04<00:00,  5.21it/s]


Epoch 35: Macro=0.8997, Micro=0.9000
早停
Fold 4 最佳 Macro-F1: 0.9302

========== v2 五折平均 Macro-F1: 0.9097 ==========


In [5]:
# ========== 预测 + TTA + 生成 submission（自包含版本）==========
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

# ========== 配置 ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 5
IMG_SIZE = 224
N_FOLDS = 5
WORK_DIR = "/kaggle/working"

# ========== 路径 ==========
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")

# ========== 类别名（必须和训练时顺序一致）==========
class_names = sorted(os.listdir(TRAIN_DIR))
print("类别顺序:", class_names)

# ========== 验证集 transform ==========
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ========== 模型构建（预测时不需要下载预训练权重）==========
def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

# ========== TTA 预测 ==========
def predict_with_tta(model, files):
    model.eval()
    probs = []
    with torch.no_grad():
        for f in tqdm(files, desc="Predict"):
            img = Image.open(f).convert("RGB")
            x = val_transform(img).unsqueeze(0).to(DEVICE)
            # 原图 + 水平翻转
            p1 = F.softmax(model(x), dim=1)
            p2 = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            probs.append(((p1 + p2) / 2).cpu().numpy())
    return np.concatenate(probs, axis=0)

# ========== 加载测试集文件名 ==========
test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)])
print(f"测试集数量: {len(test_files)}")

# ========== 5 折模型集成 + TTA ==========
ensemble_probs = np.zeros((len(test_files), NUM_CLASSES))
for fold in range(N_FOLDS):
    model_path = f"{WORK_DIR}/model_v2_fold{fold}.pth"
    model = build_model()
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    probs = predict_with_tta(model, test_files)
    ensemble_probs += probs
    print(f"Fold {fold} 预测完成")
ensemble_probs /= N_FOLDS
preds = ensemble_probs.argmax(axis=1)

# ========== 生成 submission（严格匹配示例格式）==========
submission = pd.DataFrame({
    "ID": [os.path.basename(f) for f in test_files],
    "Category": [class_names[i] for i in preds]
})
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)

# ========== 检查输出 ==========
print("\n========== 提交文件检查 ==========")
print(submission.head(10))
print(f"\n总行数: {len(submission)}")
print(f"列名: {list(submission.columns)}")
print(f"\n类别分布:\n{submission['Category'].value_counts()}")
print(f"\n文件已保存到: {WORK_DIR}/submission.csv")

类别顺序: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']
测试集数量: 378


Predict: 100%|██████████| 378/378 [00:07<00:00, 51.51it/s]


Fold 0 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 52.92it/s]


Fold 1 预测完成


Predict: 100%|██████████| 378/378 [00:06<00:00, 54.73it/s]


Fold 2 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.25it/s]


Fold 3 预测完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.07it/s]

Fold 4 预测完成

========== 提交文件检查 ==========
               ID           Category
0  abkieurfrr.png         Sugar beet
1  ablzkpojla.png         Sugar beet
2  acmbgjflct.png   Loose Silky-bent
3  afhcnytjtd.png   Loose Silky-bent
4  ahoyeyvosg.png   Loose Silky-bent
5  aijmdaocym.png   Loose Silky-bent
6  ajjwrhipgh.png   Loose Silky-bent
7  apjrnavrnr.png         Sugar beet
8  arkzcnxddq.png  Scentless Mayweed
9  armpwctjip.png         Sugar beet

总行数: 378
列名: ['ID', 'Category']

类别分布:
Category
Loose Silky-bent     94
Scentless Mayweed    93
Sugar beet           82
Black-grass          72
Common wheat         37
Name: count, dtype: int64

文件已保存到: /kaggle/working/submission.csv


In [6]:
# ========== v1 + v2 模型集成 ==========
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 5
IMG_SIZE = 224
WORK_DIR = "/kaggle/working"
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")

class_names = sorted(os.listdir(TRAIN_DIR))

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

def predict_with_tta(model, files):
    model.eval()
    probs = []
    with torch.no_grad():
        for f in tqdm(files, desc="Predict"):
            img = Image.open(f).convert("RGB")
            x = val_transform(img).unsqueeze(0).to(DEVICE)
            p1 = F.softmax(model(x), dim=1)
            p2 = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            probs.append(((p1 + p2) / 2).cpu().numpy())
    return np.concatenate(probs, axis=0)

test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)])
print(f"测试集数量: {len(test_files)}")

# ---- 加载 v1 + v2 共 10 个模型 ----
model_paths = []
for fold in range(5):
    model_paths.append(f"{WORK_DIR}/model_fold{fold}.pth")       # v1
    model_paths.append(f"{WORK_DIR}/model_v2_fold{fold}.pth")    # v2

ensemble_probs = np.zeros((len(test_files), NUM_CLASSES))
for path in model_paths:
    if not os.path.exists(path):
        print(f"⚠️ 缺失: {path}")
        continue
    model = build_model()
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    probs = predict_with_tta(model, test_files)
    ensemble_probs += probs
    print(f"✅ {os.path.basename(path)} 完成")

ensemble_probs /= len(model_paths)
preds = ensemble_probs.argmax(axis=1)

submission = pd.DataFrame({
    "ID": [os.path.basename(f) for f in test_files],
    "Category": [class_names[i] for i in preds]
})
submission.to_csv(f"{WORK_DIR}/submission_ensemble.csv", index=False)
print("\n已生成 submission_ensemble.csv")
print(submission['Category'].value_counts())

测试集数量: 378


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.51it/s]


✅ model_fold0.pth 完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.73it/s]


✅ model_v2_fold0.pth 完成


Predict: 100%|██████████| 378/378 [00:07<00:00, 53.67it/s]


... (11 outputs left out)

Predict: 100%|██████████| 378/378 [00:07<00:00, 52.28it/s]


✅ model_fold4.pth 完成


Predict: 100%|██████████| 378/378 [00:06<00:00, 55.19it/s]

✅ model_v2_fold4.pth 完成

已生成 submission_ensemble.csv
Category
Loose Silky-bent     98
Scentless Mayweed    93
Sugar beet           80
Black-grass          68
Common wheat         39
Name: count, dtype: int64


In [7]:
#v3测试
# ========== 配置 ==========
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", DEVICE)

NUM_CLASSES = 5
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 40
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 10
N_FOLDS = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
WORK_DIR = "/kaggle/working"

# ========== 加载数据（同上）==========
def load_train_data(data_dir):
    filenames, labels = [], []
    classes = sorted(os.listdir(data_dir))
    for label, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        for fname in os.listdir(cls_dir):
            filenames.append(os.path.join(cls_dir, fname))
            labels.append(label)
    return filenames, labels, classes

train_files, train_labels, class_names = load_train_data(TRAIN_DIR)
print(f"总样本数: {len(train_files)}, 类别: {class_names}")

# ========== 数据增强 ==========
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class PlantDataset(Dataset):
    def __init__(self, filenames, labels, transform=None):
        self.filenames = filenames
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, idx):
        img = Image.open(self.filenames[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# ========== ★ 关键改动：EfficientNet-B0 ==========
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

# ========== 单折训练 ==========
def train_one_fold(fold, train_f, train_l, val_f, val_l):
    print(f"\n===== Fold {fold} =====")
    train_ds = PlantDataset(train_f, train_l, train_transform)
    val_ds = PlantDataset(val_f, val_l, val_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # 阶段1：冻结 backbone，只训练 classifier
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=1e-4)

    for epoch in range(EPOCHS_HEAD):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Head] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    # 阶段2：解冻全部微调
    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE)

    best_f1, best_state, wait = 0, None, 0
    for epoch in range(EPOCHS_FINETUNE):
        model.train()
        for imgs, labels in tqdm(train_loader, desc=f"[Fine] E{epoch+1}"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE)
                p = model(imgs).argmax(dim=1).cpu().numpy()
                preds.extend(p); gts.extend(labels.numpy())
        macro_f1 = f1_score(gts, preds, average='macro')
        print(f"Epoch {epoch+1}: Macro={macro_f1:.4f}")
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print("早停"); break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"{WORK_DIR}/model_eff_fold{fold}.pth")
    print(f"Fold {fold} 最佳 Macro-F1: {best_f1:.4f}")
    return best_f1

# ========== 5 折 ==========
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(train_files, train_labels)):
    tr_f = [train_files[i] for i in tr_idx]
    tr_l = [train_labels[i] for i in tr_idx]
    va_f = [train_files[i] for i in val_idx]
    va_l = [train_labels[i] for i in val_idx]
    fold_scores.append(train_one_fold(fold, tr_f, tr_l, va_f, va_l))

print(f"\n========== EfficientNet-B0 五折平均 Macro-F1: {np.mean(fold_scores):.4f} ==========")

使用设备: cuda
总样本数: 500, 类别: ['Black-grass', 'Common wheat', 'Loose Silky-bent', 'Scentless Mayweed', 'Sugar beet']

===== Fold 0 =====
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 136MB/s] 
[Fine] E1: 100%|██████████| 25/25 [00:04<00:00,  5.56it/s]


Epoch 1: Macro=0.7793


[Fine] E2: 100%|██████████| 25/25 [00:04<00:00,  5.21it/s]


Epoch 2: Macro=0.8079


[Fine] E3: 100%|██████████| 25/25 [00:04<00:00,  5.43it/s]


... (203 outputs left out)

[Fine] E22: 100%|██████████| 25/25 [00:04<00:00,  5.32it/s]


Epoch 22: Macro=0.8987


[Fine] E23: 100%|██████████| 25/25 [00:04<00:00,  5.10it/s]


Epoch 23: Macro=0.8896
早停
Fold 4 最佳 Macro-F1: 0.9297

========== EfficientNet-B0 五折平均 Macro-F1: 0.8892 ==========


In [8]:
#v3测试

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 5
IMG_SIZE = 224
WORK_DIR = "/kaggle/working"
BASE = "/kaggle/input/competitions/2026task1/dataset-for-task2/dataset-for-task2"
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
class_names = sorted(os.listdir(TRAIN_DIR))

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def build_resnet():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, NUM_CLASSES))
    return model.to(DEVICE)

def build_effnet():
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, NUM_CLASSES))
    return model.to(DEVICE)

def predict_with_tta(model, files):
    model.eval()
    probs = []
    with torch.no_grad():
        for f in tqdm(files, desc="Predict"):
            img = Image.open(f).convert("RGB")
            x = val_transform(img).unsqueeze(0).to(DEVICE)
            p1 = F.softmax(model(x), dim=1)
            p2 = F.softmax(model(torch.flip(x, dims=[3])), dim=1)
            probs.append(((p1 + p2) / 2).cpu().numpy())
    return np.concatenate(probs, axis=0)

test_files = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)])
print(f"测试集: {len(test_files)}")

# 15 个模型清单
model_list = []
for fold in range(5):
    model_list.append((build_resnet, f"{WORK_DIR}/model_fold{fold}.pth"))       # v1 ResNet
    model_list.append((build_resnet, f"{WORK_DIR}/model_v2_fold{fold}.pth"))    # v2 ResNet
    model_list.append((build_effnet, f"{WORK_DIR}/model_eff_fold{fold}.pth"))   # EfficientNet

ensemble_probs = np.zeros((len(test_files), NUM_CLASSES))
count = 0
for build_fn, path in model_list:
    if not os.path.exists(path):
        print(f"⚠️ 缺失: {path}"); continue
    model = build_fn()
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    ensemble_probs += predict_with_tta(model, test_files)
    count += 1
    print(f"✅ {os.path.basename(path)}")

ensemble_probs /= count
preds = ensemble_probs.argmax(axis=1)
print(f"\n参与集成的模型数: {count}")

submission = pd.DataFrame({
    "ID": [os.path.basename(f) for f in test_files],
    "Category": [class_names[i] for i in preds]
})
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)
print("已生成 submission.csv")
print(submission['Category'].value_counts())

测试集: 378


Predict: 100%|██████████| 378/378 [00:09<00:00, 38.90it/s]


✅ model_fold0.pth


Predict: 100%|██████████| 378/378 [00:07<00:00, 52.25it/s]


✅ model_v2_fold0.pth


Predict: 100%|██████████| 378/378 [00:11<00:00, 33.82it/s]


... (21 outputs left out)

Predict: 100%|██████████| 378/378 [00:06<00:00, 54.44it/s]


✅ model_v2_fold4.pth


Predict: 100%|██████████| 378/378 [00:11<00:00, 33.60it/s]

✅ model_eff_fold4.pth

参与集成的模型数: 15
已生成 submission.csv
Category
Loose Silky-bent     106
Scentless Mayweed     92
Sugar beet            81
Black-grass           64
Common wheat          35
Name: count, dtype: int64
